## CODE TO SCRAPE ALL ~340 FILES
AND 
## CODE TO SELECT ONLY 15 OF VALID FILES SCRAPED

In [ ]:
import pandas as pd
import requests
import json
import time
from bs4 import BeautifulSoup

df = pd.read_csv("final_gold_standard.csv")

def scrape_pmc(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    
    if "pubmed.ncbi.nlm.nih.gov" in url:
        pubmed_id = url.strip("/").split("/")[-1]
        search_url = f"https://www.ncbi.nlm.nih.gov/pmc/utils/idconv/v1.0/?ids={pubmed_id}&format=json"
        try:
            r = requests.get(search_url, headers=headers)
            data = r.json()
            pmc_id = data["records"][0].get("pmcid")
            if pmc_id:
                url = f"https://pmc.ncbi.nlm.nih.gov/articles/{pmc_id}/"
        except:
            pass

    try:
        r = requests.get(url, headers=headers, timeout=15)
        soup = BeautifulSoup(r.text, "html.parser")

        for tag in soup.find_all(["table", "figure", "sup"]):
            tag.decompose()

        article = soup.find("div", class_="article-body") or \
                  soup.find("div", id="body") or \
                  soup.find("main") or \
                  soup.find("article")

        if article:
            return article.get_text(separator=" ", strip=True)
        else:
            return soup.get_text(separator=" ", strip=True)[:50000]

    except Exception as e:
        return f"ERROR: {e}"

results = []
failed = []

for _, row in df.iterrows():
    title = row["Title"]
    link = str(row["Link (Use DOI or Title if missing)"])
    print(f"Scraping: {title[:60]}...")

    if link == "nan" or not link.startswith("http"):
        print("  No link, skipping")
        continue

    text = scrape_pmc(link)
    time.sleep(2)

    char_len = len(text)
    is_usable = char_len > 5000 and "Enable JavaScript" not in text and "Just a moment" not in text

    print(f"  {'OK' if is_usable else 'FAILED'} — {char_len} chars")

    entry = {
        "title": title,
        "link": link,
        "disease": row["Disease"],
        "taxa_enriched": row["KeyTaxa_Enriched"],
        "taxa_depleted": row["KeyTaxa_Depleted"],
        "in_gold_standard": row["InGoldStandard"],
        "char_len": char_len,
        "usable": is_usable,
        "text": text
    }

    if is_usable:
        results.append(entry)
    else:
        failed.append(title)

print(f"\nUsable: {len(results)} / {len(df)}")
print(f"Failed: {len(failed)}")

# Save all usable
with open("all_usable_papers.json", "w") as f:
    json.dump(results, f, indent=2)

# Sample 10 yes 5 no from usable
usable_df = pd.DataFrame([{
    "title": r["title"],
    "in_gold_standard": r["in_gold_standard"]
} for r in results])

yes_pool = usable_df[usable_df["in_gold_standard"] == "Yes"]
no_pool = usable_df[usable_df["in_gold_standard"] == "No"]

print(f"\nUsable Yes: {len(yes_pool)}, Usable No: {len(no_pool)}")

yes_sample = yes_pool.sample(n=min(10, len(yes_pool)), random_state=42)
no_sample = no_pool.sample(n=min(5, len(no_pool)), random_state=42)

selected_titles = set(yes_sample["title"].tolist() + no_sample["title"].tolist())
final_15 = [r for r in results if r["title"] in selected_titles]

with open("gold_standard_final_15.json", "w") as f:
    json.dump(final_15, f, indent=2)

print(f"Saved {len(final_15)} papers to gold_standard_final_15.json")

Scraping: Dysbiosis characteristics of gut microbiota in cerebral infa...
  OK — 47404 chars
Scraping: Gut Microbiota and Fecal Metabolites Associated With Neuroco...
  OK — 41253 chars
Scraping: Hemorrhagic transformation in patients with large-artery ath...
  FAILED — 58 chars
Scraping: Rett Syndrome: A Focus on Gut Microbiota...
  FAILED — 224 chars
Scraping: The Alterations of Gut Microbiome and Lipid Metabolism in Pa...
  OK — 42573 chars
Scraping: Distinctive Gut Microbiota Alteration Is Associated with Pos...
  FAILED — 58 chars
Scraping: Comparison of the effects of probiotics, rifaximin, and lact...
  OK — 47747 chars
Scraping: Oral Pathobiont Streptococcus Anginosus Is Enriched in the G...
  OK — 8366 chars
Scraping: Integrated Traditional Chinese Medicine Improves Functional ...
  OK — 52344 chars
Scraping: The gut microbial signatures of patients with lacunar cerebr...
  FAILED — 58 chars
Scraping: Altered Gut Microbiota and Plasma Metabolome Profiles Charac...
  OK — 50000

### QUICK VERIFICATION

In [10]:
import json

with open("gold_standard_final_15.json", "r") as f:
    papers = json.load(f)

for p in papers:
    print(f"[{p['in_gold_standard']}] {p['title'][:70]} — {p['char_len']} chars")

[No] Intestinal flora induces depression by mediating the dysregulation of  — 59748 chars
[Yes] Gut microbes exacerbate systemic inflammation and behavior disorders i — 93335 chars
[Yes] Alterations in gut microbiota and metabolomic profiles in acute stroke — 67570 chars
[No] Gut microbiome dysbiosis across early Parkinson's disease, REM sleep b — 79012 chars
[Yes] The gut microbiota in multiple sclerosis varies with disease activity. — 80834 chars
[No] Gut Microbial Ecosystem in Parkinson Disease: New Clinicobiological In — 9977 chars
[Yes] Dysbiosis of gut microbiota in a selected population of Parkinson's pa — 5985 chars
[No] Gut microbiota distinguishes aging hispanics with Alzheimer's disease: — 75825 chars
[No] Examining the complex Interplay between gut microbiota abundance and s — 49363 chars
[Yes] Characterizing Gut Microbiota in Older Chinese Adults with Cognitive I — 6996 chars
[Yes] Gut microbiome, cognitive function and brain structure: a multi-omics  — 64883 chars
[Yes] R

# CHECKING HOW MANY PAPERS ACTUALLY ARE USBALE FROM THE TOTAL SET

In [1]:
import json
import pandas as pd
import re

# pandas handles NaN in JSON natively, json.load chokes on it
df = pd.read_json("all_usable_papers.json")

# normalize: treat empty string, "nan", whitespace, NaN all as missing
def is_empty(x):
    if pd.isna(x):
        return True
    s = str(x).strip().lower()
    return s in ("", "nan", "none", "n/a")

df["has_enriched"] = ~df["taxa_enriched"].apply(is_empty)
df["has_depleted"] = ~df["taxa_depleted"].apply(is_empty)
df["has_annotation"] = df["has_enriched"] | df["has_depleted"]

# filter to only annotated papers
annotated = df[df["has_annotation"]].copy().reset_index(drop=True)

print(f"Total scraped: {len(df)}")
print(f"With annotations: {len(annotated)}")
print(f"  enriched only: {(annotated['has_enriched'] & ~annotated['has_depleted']).sum()}")
print(f"  depleted only: {(~annotated['has_enriched'] & annotated['has_depleted']).sum()}")
print(f"  both: {(annotated['has_enriched'] & annotated['has_depleted']).sum()}")

# EDA on text content
def section_check(text):
    if pd.isna(text):
        return {"chars": 0, "has_methods": False, "has_results": False, "has_discussion": False, "likely_full": False}
    t = str(text)
    has_methods = bool(re.search(r'\b(methods|materials and methods)\b', t, re.I))
    has_results = bool(re.search(r'\bresults\b', t, re.I))
    has_discussion = bool(re.search(r'\bdiscussion\b', t, re.I))
    return {
        "chars": len(t),
        "has_methods": has_methods,
        "has_results": has_results,
        "has_discussion": has_discussion,
        "likely_full": has_methods and has_results and has_discussion,
    }

eda = pd.DataFrame([section_check(t) for t in annotated["text"]])
annotated = pd.concat([annotated.drop(columns=[c for c in eda.columns if c in annotated.columns]), eda], axis=1)

print(f"\n--- text length on annotated papers ---")
print(annotated["chars"].describe().round(0))

print(f"\n--- full text detection ---")
print(f"Likely full text (has methods+results+discussion): {annotated['likely_full'].sum()}/{len(annotated)}")
print(f"Has results section:    {annotated['has_results'].sum()}/{len(annotated)}")
print(f"Has methods section:    {annotated['has_methods'].sum()}/{len(annotated)}")
print(f"Has discussion section: {annotated['has_discussion'].sum()}/{len(annotated)}")

# papers under 5k chars are almost certainly abstract-only or scrape failures
print(f"\nUnder 5k chars (likely abstract/failed): {(annotated['chars'] < 5000).sum()}")
print(f"5k-15k (partial / abstract+intro):       {((annotated['chars'] >= 5000) & (annotated['chars'] < 15000)).sum()}")
print(f"15k+ (likely full paper):                {(annotated['chars'] >= 15000).sum()}")

# save the clean filtered set for downstream use
annotated.to_json("annotated_papers.json", orient="records", indent=2)
print(f"\nSaved {len(annotated)} annotated papers to annotated_papers.json")

# show me a quick sample to sanity-check
print(f"\n--- sample of 5 annotated papers ---")
for _, row in annotated.sample(min(5, len(annotated)), random_state=0).iterrows():
    print(f"  [{row['chars']:>6} chars | full={row['likely_full']}] {row['title'][:70]}")

Total scraped: 250
With annotations: 88
  enriched only: 8
  depleted only: 3
  both: 77

--- text length on annotated papers ---
count        88.0
mean      50206.0
std       25658.0
min        5136.0
25%       38635.0
50%       49852.0
75%       67776.0
max      100290.0
Name: chars, dtype: float64

--- full text detection ---
Likely full text (has methods+results+discussion): 72/88
Has results section:    83/88
Has methods section:    82/88
Has discussion section: 72/88

Under 5k chars (likely abstract/failed): 0
5k-15k (partial / abstract+intro):       15
15k+ (likely full paper):                73

Saved 88 annotated papers to annotated_papers.json

--- sample of 5 annotated papers ---
  [ 42573 chars | full=True] The Alterations of Gut Microbiome and Lipid Metabolism in Patients wit
  [ 93335 chars | full=True] Gut microbes exacerbate systemic inflammation and behavior disorders i
  [ 67241 chars | full=True] Disturbed microbial ecology in Alzheimer's disease: evidence from the 


In [2]:
import re

def rigorous_full_text_check(text):
    if pd.isna(text) or len(str(text)) < 8000:
        return {"is_full_paper": False, "score": 0, "signals": {}}
    t = str(text)
    
    # section headers as structural (capitalized standalone, or after newline)
    section_patterns = {
        "intro_header":      r'(?:^|\n)\s*(?:1\.?\s*)?(?:INTRODUCTION|Introduction)\b',
        "methods_header":    r'(?:^|\n)\s*(?:2\.?\s*)?(?:METHODS|Methods|MATERIALS AND METHODS|Materials and Methods)\b',
        "results_header":    r'(?:^|\n)\s*(?:3\.?\s*)?(?:RESULTS|Results)\b',
        "discussion_header": r'(?:^|\n)\s*(?:4\.?\s*)?(?:DISCUSSION|Discussion)\b',
    }
    headers_found = {k: bool(re.search(p, t)) for k, p in section_patterns.items()}
    
    # quantitative signals
    n_pvalues = len(re.findall(r'[pP]\s*[=<>]\s*0?\.\d+', t))
    n_sample_sizes = len(re.findall(r'\bn\s*=\s*\d+', t))
    n_figures = len(re.findall(r'\b(?:Fig(?:ure|\.)?|Table)\s+\d+', t))
    n_citations = len(re.findall(r'\(\s*[A-Z][a-z]+(?:\s+et\s+al\.?)?,?\s*\d{4}\)|\[\d+(?:[,\-]\s*\d+)*\]', t))
    
    # methods-specific tooling vocabulary
    methods_terms = ['QIIME', 'DADA2', 'Illumina', 'MiSeq', '16S', 'rRNA', 'OTU', 'ASV', 'Bray-Curtis', 'LEfSe', 'PERMANOVA', 'Wilcoxon']
    n_methods_terms = sum(1 for term in methods_terms if term.lower() in t.lower())
    
    signals = {
        "headers_found": sum(headers_found.values()),
        "n_pvalues": n_pvalues,
        "n_sample_sizes": n_sample_sizes,
        "n_figure_refs": n_figures,
        "n_citations": n_citations,
        "n_methods_terms": n_methods_terms,
        "char_len": len(t),
    }
    
    # scoring: each strong signal adds 1
    score = 0
    if signals["headers_found"] >= 3: score += 2  # intro/methods/results/discussion all present
    if signals["n_pvalues"] >= 5: score += 1
    if signals["n_figure_refs"] >= 3: score += 1
    if signals["n_citations"] >= 10: score += 1
    if signals["n_methods_terms"] >= 3: score += 1
    if signals["char_len"] >= 20000: score += 1
    
    # high confidence threshold
    is_full = score >= 5
    return {"is_full_paper": is_full, "score": score, "signals": signals}

# apply
checks = annotated["text"].apply(rigorous_full_text_check)
annotated["full_score"] = checks.apply(lambda x: x["score"])
annotated["is_full_strict"] = checks.apply(lambda x: x["is_full_paper"])

print(f"Strict full paper (score >= 5): {annotated['is_full_strict'].sum()}/{len(annotated)}")
print(f"\nScore distribution:")
print(annotated["full_score"].value_counts().sort_index())

# show borderline cases (score 3-4) so you can eyeball them
print(f"\n--- borderline papers (score 3-4) ---")
for _, row in annotated[annotated["full_score"].between(3, 4)].iterrows():
    sigs = rigorous_full_text_check(row["text"])["signals"]
    print(f"  score={row['full_score']} | chars={sigs['char_len']:>6} | hdrs={sigs['headers_found']} | p-vals={sigs['n_pvalues']:>3} | figs={sigs['n_figure_refs']:>3} | cites={sigs['n_citations']:>3} | {row['title'][:55]}")

# final filtered set
strict = annotated[annotated["is_full_strict"]].copy()
strict.to_json("annotated_full_papers.json", orient="records", indent=2)
print(f"\nSaved {len(strict)} full-text annotated papers to annotated_full_papers.json")

Strict full paper (score >= 5): 3/88

Score distribution:
full_score
0    13
1     2
2     1
3    12
4    57
5     3
Name: count, dtype: int64

--- borderline papers (score 3-4) ---
  score=4 | chars= 41253 | hdrs=0 | p-vals= 18 | figs= 13 | cites=  0 | Gut Microbiota and Fecal Metabolites Associated With Ne
  score=4 | chars= 42573 | hdrs=0 | p-vals= 15 | figs= 13 | cites=  0 | The Alterations of Gut Microbiome and Lipid Metabolism 
  score=4 | chars= 47747 | hdrs=0 | p-vals=  9 | figs= 24 | cites=  0 | Comparison of the effects of probiotics, rifaximin, and
  score=4 | chars= 52344 | hdrs=0 | p-vals=  9 | figs= 23 | cites=  0 | Integrated Traditional Chinese Medicine Improves Functi
  score=4 | chars= 50000 | hdrs=0 | p-vals= 18 | figs= 15 | cites=  0 | Altered Gut Microbiota and Plasma Metabolome Profiles C
  score=4 | chars= 45503 | hdrs=0 | p-vals= 37 | figs= 17 | cites=  0 | Altered gut microbiota in individuals with episodic and
  score=4 | chars= 46449 | hdrs=0 | p-vals= 15 | f

In [3]:
def fixed_check(text):
    if pd.isna(text) or len(str(text)) < 8000:
        return 0
    t = str(text)
    score = 0
    if len(t) >= 20000: score += 2
    if len(re.findall(r'[pP]\s*[=<>]\s*0?\.\d+', t)) >= 5: score += 1
    if len(re.findall(r'\b(?:Fig(?:ure|\.)?|Table)\s+\d+', t)) >= 5: score += 1
    methods_terms = ['QIIME', 'DADA2', 'Illumina', 'MiSeq', '16S', 'rRNA', 'OTU', 'ASV', 'LEfSe', 'PERMANOVA', 'Wilcoxon', 'Bray-Curtis']
    if sum(1 for term in methods_terms if term.lower() in t.lower()) >= 2: score += 1
    return score

annotated["fixed_score"] = annotated["text"].apply(fixed_check)
print(annotated["fixed_score"].value_counts().sort_index())
print(f"\nUsable (score >= 3): {(annotated['fixed_score'] >= 3).sum()}")

usable = annotated[annotated["fixed_score"] >= 3].copy()
usable.to_json("usable_papers.json", orient="records", indent=2)
print(f"Saved {len(usable)} usable papers")

fixed_score
0    12
1     3
3     1
4    14
5    58
Name: count, dtype: int64

Usable (score >= 3): 73
Saved 73 usable papers


In [4]:
usable = annotated[annotated["fixed_score"] >= 4].copy().reset_index(drop=True)
usable.to_json("usable_papers.json", orient="records", indent=2)
print(f"Kept {len(usable)} papers (score >= 4)")

Kept 72 papers (score >= 4)


## Building the locked test set

### Why we're rebuilding
Prior eval runs (15 papers) had inconsistent annotations — papers were tagged with
a Yes/No "in_gold_standard" flag based on subjective notes, which made scoring
incoherent. Some "No" papers had real annotations; some had empty annotation columns.
The Yes/No flag is being dropped entirely. We're starting over with a cleanly
filtered set of papers that are guaranteed to be (a) full text, not abstracts,
and (b) actually annotated by the labmate.

### Pipeline summary
1. Started with 250 scraped papers (`all_usable_papers.json`).
2. Filtered to 88 papers with non-empty taxa annotations (enriched OR depleted).
3. Scored each annotated paper for full-text completeness (see scoring below).
4. Kept only papers with score ≥ 4 → 72 high-confidence full papers.
5. Locked 15 of those as the held-out test set. The remaining 57 are reserved
   for future use but are not the focus right now.

### How we score full-text completeness
The original scraper used `get_text(separator=" ")` which flattened newlines into
spaces. Section headers like "Methods" or "Results" still appear as words in the
text but no longer as structural elements, so a header-based detector returns
zero for every paper. Instead we score papers using content-density signals
that survive the flattening:

- **Length (≥20k chars)**: full papers are 30–100k chars; abstracts are 1–3k. (+2)
- **p-values (≥5)**: real papers report dozens of statistical tests
  (`p < 0.05`, `p = 0.03`). Abstracts mention 0–2. (+1)
- **Figure/table refs (≥5)**: full papers cite "Figure 1", "Table 2"
  repeatedly throughout. (+1)
- **Methods vocabulary (≥2 terms)**: tools like QIIME, DADA2, Illumina, 16S,
  LEfSe, PERMANOVA only appear in real methods sections. (+1)

Max score is 5. Higher = more confident the scrape captured a real full paper.

### Score distribution (annotated papers, n=88)
| Score | Count | Interpretation |
|------:|------:|----------------|
| 0     | 12    | Junk: scrape failures, paywalled stubs |
| 1     | 3     | Junk: too short / abstract-only |
| 3     | 1     | Borderline, excluded for safety |
| 4     | 14    | Confident full paper |
| 5     | 58    | Confident full paper, citation format survived scraping |

### Decision
- Keep only score ≥ 4 papers → **72 confident full papers** with annotations.
- Lock **15 of those as the test set** (`test_set_v2.json`), stratified by disease
  where possible.
- The other 57 are kept aside in `train_pool.json` for later use.
- All eval runs from this point forward use `test_set_v2.json`. Prior eval
  results (Llama 3.3, Qwopus v1, Qwopus v3 simple, Qwopus v3 goated) are
  retired — they were measured on a noisy test set and aren't comparable
  to anything going forward.

### What's next
1. Rewrite the eval pipeline with three fixes:
   a. Drop the Yes/No flag entirely.
   b. Hungarian matching so each predicted taxon matches at most one gold taxon.
   c. NCBI Taxonomy normalization via ete3 so genus/family/species relationships
      get partial credit instead of being marked wrong.
2. Run Qwopus v3 with the goated XML + grammar prompt on the new test set
   as the real baseline.
3. Compare against Qwopus3.6-27B-v1-preview and Qwen3.6-35B-A3B as quick
   sanity checks before committing to one model long-term.

In [5]:
import re
import numpy as np

def score_paper(text):
    if pd.isna(text) or len(str(text)) < 8000:
        return 0
    t = str(text)
    score = 0
    if len(t) >= 20000:
        score += 2
    if len(re.findall(r'[pP]\s*[=<>]\s*0?\.\d+', t)) >= 5:
        score += 1
    if len(re.findall(r'\b(?:Fig(?:ure|\.)?|Table)\s+\d+', t)) >= 5:
        score += 1
    methods_terms = ['QIIME', 'DADA2', 'Illumina', 'MiSeq', '16S', 'rRNA',
                     'OTU', 'ASV', 'LEfSe', 'PERMANOVA', 'Wilcoxon', 'Bray-Curtis']
    if sum(1 for term in methods_terms if term.lower() in t.lower()) >= 2:
        score += 1
    return score

annotated["score"] = annotated["text"].apply(score_paper)
print("Score distribution:")
print(annotated["score"].value_counts().sort_index())

usable = annotated[annotated["score"] >= 4].copy().reset_index(drop=True)
print(f"\nUsable papers (score >= 4): {len(usable)}")

# stratify test set by disease
np.random.seed(42)
usable["disease_clean"] = usable["disease"].fillna("Unknown").astype(str).str.strip()

test_ids = []
for disease, group in usable.groupby("disease_clean"):
    n_pick = max(1, round(len(group) * 15 / len(usable)))
    test_ids.extend(group.sample(min(n_pick, len(group)), random_state=42).index.tolist())

# trim to exactly 15
if len(test_ids) > 15:
    test_ids = list(np.random.choice(test_ids, 15, replace=False))

test_set = usable.loc[test_ids].reset_index(drop=True)
holdout = usable.drop(test_ids).reset_index(drop=True)

test_set.to_json("test_set_v2.json", orient="records", indent=2)
holdout.to_json("holdout_pool.json", orient="records", indent=2)

print(f"\nTest set: {len(test_set)} papers (LOCKED, do not modify)")
print(f"Holdout pool: {len(holdout)} papers")
print(f"\nTest set disease distribution:")
print(test_set["disease_clean"].value_counts())

Score distribution:
score
0    12
1     3
3     1
4    14
5    58
Name: count, dtype: int64

Usable papers (score >= 4): 72

Test set: 15 papers (LOCKED, do not modify)
Holdout pool: 57 papers

Test set disease distribution:
disease_clean
Parkinson’s disease (PD)               3
Alzheimer’s disease (AD)               2
Alzheimer’s disease (AD), Dementia     2
Stroke                                 2
Spinal Muscular Atrophy (SMA)          1
Epilepsy                               1
Cognitive Impairment                   1
Other                                  1
Multiple sclerosis (MS)                1
Amyotrophic lateral sclerosis (ALS)    1
Name: count, dtype: int64


### Files generated by this notebook

| File | Contents | Use |
|------|----------|-----|
| `annotated_papers.json` | All 88 papers with non-empty taxa annotations (any score). | Intermediate. Drop. |
| `annotated_fullpapers.json` | Strict v1 filter (score ≥ 5 on broken header check) — only 3 papers. | Stale, broken scoring. Drop. |
| `usable_papers.json` | 73 papers from the v1 fix (score ≥ 3). | Stale, superseded. Drop. |
| `test_set_v2.json` | **15 papers locked as held-out test set, stratified by disease.** | **Use for all eval runs going forward. Do not modify.** |
| `holdout_pool.json` | 57 papers (score ≥ 4) reserved for future use. | Keep aside, don't touch during eval development. |

**Active files: `test_set_v2.json` (15) and `holdout_pool.json` (72-15).** The other three are intermediate artifacts from earlier filtering iterations and can be deleted.